# Notebook 02 — Preprocessing and Tiling

**Purpose:** Convert all data to common projection, align SAR imagery with labels, tile into ML-ready patches.

**Inputs:** `data/raw/` from Notebook 01  
**Outputs:**
- `data/processed/sar_tiles/` — 256×256 float32 SAR patches
- `data/processed/label_tiles/` — 256×256 uint8 label patches
- `data/processed/tile_metadata.csv`
- `data/splits/split_v1.json`

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else '.')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from rasterio.warp import reproject, Resampling, calculate_default_transform
from rasterio.windows import Window
from rasterio import features as rio_features
from pathlib import Path
from collections import Counter
from tqdm.notebook import tqdm

from src.utils import (
    RAW_DIR, PROCESSED_DIR, SPLITS_DIR, FIGURES_DIR,
    read_geotiff, write_geotiff, save_array,
    TERRAIN_CLASSES, CLASS_COLORS, NUM_CLASSES,
    TITAN_SIMPLE_CYLINDRICAL_WKT, get_logger,
)

log = get_logger('02_preprocessing')

SAR_TILES_DIR = PROCESSED_DIR / 'sar_tiles'
LABEL_TILES_DIR = PROCESSED_DIR / 'label_tiles'
NLDSAR_TILES_DIR = PROCESSED_DIR / 'nldsar_tiles'

for d in [SAR_TILES_DIR, LABEL_TILES_DIR, NLDSAR_TILES_DIR, SPLITS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

## 1. Load and Inspect SAR Mosaic

In [ ]:
sar_path = RAW_DIR / 'Titan_SAR_HiSAR_Global_Mosaic_351m.tif'
assert sar_path.exists(), f"SAR mosaic not found at {sar_path}. Run Notebook 01 first."

with rasterio.open(sar_path) as src:
    sar_profile = src.profile.copy()
    sar_meta = {
        'crs': str(src.crs),
        'res': src.res,
        'shape': (src.height, src.width),
        'bounds': src.bounds,
        'dtype': str(src.dtypes[0]),
        'nodata': src.nodata,
        'transform': src.transform,
    }
    
    # Read a sample region for quick inspection
    # Center of Shangri-La dune field (~10°S, 170°W → ~170°E in 0-360 convention)
    sample_window = Window(col_off=src.width//3, row_off=src.height//3,
                           width=512, height=512)
    sar_sample = src.read(1, window=sample_window)

print("SAR Mosaic Properties:")
for k, v in sar_meta.items():
    print(f"  {k}: {v}")

print(f"\nSample stats: min={sar_sample.min():.4f}, max={sar_sample.max():.4f}, "
      f"mean={sar_sample.mean():.4f}, dtype={sar_sample.dtype}")

fig, ax = plt.subplots(1, 1, figsize=(6, 6))
ax.imshow(sar_sample, cmap='gray', vmin=np.percentile(sar_sample[sar_sample > 0], 2),
          vmax=np.percentile(sar_sample[sar_sample > 0], 98))
ax.set_title('SAR Mosaic Sample')
plt.tight_layout()
plt.show()

## 2. Load and Inspect Geomorphological Map

In [ ]:
# Check what label data we have
geomorph_dir = RAW_DIR / 'geomorphology'

# Look for GeoTIFF labels
label_tifs = list(geomorph_dir.rglob('*.tif'))
# Look for shapefiles
label_shps = list(geomorph_dir.rglob('*.shp'))

print(f"Label GeoTIFFs found: {len(label_tifs)}")
for f in label_tifs:
    print(f"  {f}")

print(f"\nLabel shapefiles found: {len(label_shps)}")
for f in label_shps:
    print(f"  {f}")

In [ ]:
# If we have shapefiles, rasterise them to match the SAR mosaic grid
import fiona
from shapely.geometry import shape

LABEL_RASTER_PATH = PROCESSED_DIR / 'label_map_aligned.tif'

if label_tifs:
    # Use existing GeoTIFF label map
    label_path = label_tifs[0]
    log.info(f"Using GeoTIFF label map: {label_path}")
    with rasterio.open(label_path) as src:
        print(f"Label CRS: {src.crs}")
        print(f"Label shape: {src.height} x {src.width}")
        print(f"Label dtype: {src.dtypes}")
        label_data = src.read(1)
        print(f"Unique values: {np.unique(label_data)}")

elif label_shps:
    # Rasterise shapefile to match SAR mosaic grid
    label_path = label_shps[0]
    log.info(f"Rasterising shapefile: {label_path}")
    
    # Build class name → index mapping from the shapefile
    with fiona.open(label_path) as src:
        shp_crs = src.crs
        
        # Identify the classification field
        class_field = None
        for feat in src:
            for key in feat['properties']:
                if any(kw in key.lower() for kw in ['class', 'type', 'unit', 'geol', 'morph']):
                    class_field = key
                    break
            if class_field:
                break
        
        if class_field is None:
            # Use the first string field
            for key, ftype in src.schema['properties'].items():
                if 'str' in ftype:
                    class_field = key
                    break
        
        print(f"Using classification field: {class_field}")
        
        # Collect unique classes
        src_iter = iter(src)
        unique_classes = set()
        for feat in fiona.open(label_path):
            if class_field and class_field in feat['properties']:
                unique_classes.add(feat['properties'][class_field])
        
        print(f"Unique classes in shapefile: {sorted(unique_classes)}")
        
        # Map shapefile classes to our 0–5 indices
        # This mapping will need manual adjustment based on the actual field values
        CLASS_MAP = {}  # Will be populated based on actual data
        # Example mapping (adjust based on actual shapefile values):
        LOPES_MAP = {
            'plains': 0, 'undifferentiated plains': 0, 'plain': 0,
            'dunes': 1, 'dune': 1, 'sand dunes': 1, 'aeolian': 1,
            'hummocky': 2, 'mountainous': 2, 'hummocky/mountainous': 2, 'mountain': 2,
            'lakes': 3, 'seas': 3, 'lakes/seas': 3, 'lake': 3, 'mare': 3, 'lacustrine': 3,
            'labyrinth': 4, 'dissected': 4,
            'craters': 5, 'crater': 5, 'impact': 5,
        }
        
        for cls_name in unique_classes:
            matched = False
            for pattern, idx in LOPES_MAP.items():
                if pattern in str(cls_name).lower():
                    CLASS_MAP[cls_name] = idx
                    matched = True
                    break
            if not matched:
                log.warning(f"Unmapped class: '{cls_name}' — assigning to plains (0)")
                CLASS_MAP[cls_name] = 0
        
        print(f"\nClass mapping:")
        for k, v in sorted(CLASS_MAP.items(), key=lambda x: x[1]):
            print(f"  '{k}' → {v} ({TERRAIN_CLASSES[v]})")
    
    # Rasterise
    with rasterio.open(sar_path) as sar_src:
        out_shape = (sar_src.height, sar_src.width)
        out_transform = sar_src.transform
        
        shapes = []
        with fiona.open(label_path) as shp_src:
            for feat in shp_src:
                geom = shape(feat['geometry'])
                cls_val = CLASS_MAP.get(feat['properties'].get(class_field, ''), 0)
                shapes.append((geom, cls_val))
        
        log.info(f"Rasterising {len(shapes)} polygons to {out_shape}...")
        label_raster = rio_features.rasterize(
            shapes,
            out_shape=out_shape,
            transform=out_transform,
            fill=255,  # nodata
            dtype='uint8',
        )
        
        # Save aligned label raster
        label_profile = sar_src.profile.copy()
        label_profile.update(dtype='uint8', count=1, nodata=255)
        write_geotiff(LABEL_RASTER_PATH, label_raster, label_profile, dtype='uint8')
        
        print(f"\nLabel raster saved: {LABEL_RASTER_PATH}")
        print(f"Shape: {label_raster.shape}")
        unique, counts = np.unique(label_raster, return_counts=True)
        for u, c in zip(unique, counts):
            name = TERRAIN_CLASSES.get(u, 'nodata' if u == 255 else f'unknown_{u}')
            print(f"  Class {u} ({name}): {c:,} pixels ({100*c/label_raster.size:.1f}%)")

else:
    log.error(
        "No label data found! The geomorphological map is required.\n"
        "Options:\n"
        "  1. Download from USGS or paper supplementary materials\n"
        "  2. Digitise from Lopes et al. (2020) Figure 1\n"
        "  3. Create approximate labels from SAR backscatter thresholds"
    )
    raise FileNotFoundError("Label map not available. See above for options.")

## 3. Verify SAR–Label Alignment

In [ ]:
# Overlay SAR + labels in a sample region to visually confirm alignment
from matplotlib.colors import ListedColormap

# Define colourmap for terrain classes
class_cmap = ListedColormap([CLASS_COLORS[i] for i in range(NUM_CLASSES)])

with rasterio.open(sar_path) as sar_src:
    if LABEL_RASTER_PATH.exists():
        label_src_path = LABEL_RASTER_PATH
    elif label_tifs:
        label_src_path = label_tifs[0]
    else:
        raise FileNotFoundError("No label raster available")
    
    with rasterio.open(label_src_path) as lbl_src:
        # Sample a 512×512 region from the middle of the data
        row_off = sar_src.height // 3
        col_off = sar_src.width // 3
        window = Window(col_off, row_off, 512, 512)
        
        sar_patch = sar_src.read(1, window=window)
        lbl_patch = lbl_src.read(1, window=window)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# SAR
valid = sar_patch[sar_patch > 0]
vmin, vmax = (np.percentile(valid, 2), np.percentile(valid, 98)) if len(valid) > 0 else (0, 1)
axes[0].imshow(sar_patch, cmap='gray', vmin=vmin, vmax=vmax)
axes[0].set_title('SAR Backscatter')

# Labels
lbl_display = np.ma.masked_where(lbl_patch == 255, lbl_patch)
axes[1].imshow(lbl_display, cmap=class_cmap, vmin=0, vmax=NUM_CLASSES-1, interpolation='nearest')
axes[1].set_title('Terrain Labels')

# Overlay
axes[2].imshow(sar_patch, cmap='gray', vmin=vmin, vmax=vmax)
axes[2].imshow(lbl_display, cmap=class_cmap, vmin=0, vmax=NUM_CLASSES-1, alpha=0.4, interpolation='nearest')
axes[2].set_title('Overlay')

for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'alignment_check.png', dpi=150, bbox_inches='tight')
plt.show()
log.info("Visual alignment check saved to figures/alignment_check.png")

## 4. Tile into 256×256 Patches

In [ ]:
TILE_SIZE = 256
NODATA_THRESHOLD = 0.5  # Discard tiles with >50% nodata

tile_records = []

with rasterio.open(sar_path) as sar_src, rasterio.open(label_src_path) as lbl_src:
    n_rows = sar_src.height // TILE_SIZE
    n_cols = sar_src.width // TILE_SIZE
    total_tiles = n_rows * n_cols
    
    log.info(f"Tiling: {n_rows} rows × {n_cols} cols = {total_tiles} potential tiles")
    
    kept = 0
    discarded = 0
    
    for row_idx in tqdm(range(n_rows), desc='Tiling rows'):
        for col_idx in range(n_cols):
            window = Window(
                col_off=col_idx * TILE_SIZE,
                row_off=row_idx * TILE_SIZE,
                width=TILE_SIZE,
                height=TILE_SIZE,
            )
            
            sar_tile = sar_src.read(1, window=window).astype(np.float32)
            lbl_tile = lbl_src.read(1, window=window).astype(np.uint8)
            
            # Compute nodata fraction (SAR nodata = 0 or src.nodata, label nodata = 255)
            sar_nodata = sar_src.nodata if sar_src.nodata is not None else 0
            nodata_mask = (sar_tile == sar_nodata) | (lbl_tile == 255)
            nodata_frac = nodata_mask.sum() / nodata_mask.size
            
            if nodata_frac > NODATA_THRESHOLD:
                discarded += 1
                continue
            
            # Normalise SAR to [0, 1] using global min/max of valid pixels
            valid = sar_tile[~nodata_mask]
            if len(valid) == 0:
                discarded += 1
                continue
            
            # Clip extreme outliers and normalise
            p2, p98 = np.percentile(valid, [2, 98])
            sar_tile_norm = np.clip(sar_tile, p2, p98)
            if p98 > p2:
                sar_tile_norm = (sar_tile_norm - p2) / (p98 - p2)
            else:
                sar_tile_norm = np.zeros_like(sar_tile_norm)
            sar_tile_norm[nodata_mask] = 0
            
            # Tile ID
            tile_id = f"tile_{row_idx:04d}_{col_idx:04d}"
            
            # Save tiles
            np.save(SAR_TILES_DIR / f"{tile_id}.npy", sar_tile_norm)
            np.save(LABEL_TILES_DIR / f"{tile_id}.npy", lbl_tile)
            
            # Compute tile metadata
            tile_bounds = rasterio.windows.bounds(window, sar_src.transform)
            class_counts = Counter(lbl_tile[~nodata_mask].flatten())
            
            record = {
                'tile_id': tile_id,
                'row': row_idx,
                'col': col_idx,
                'lon_min': tile_bounds[0],
                'lat_min': tile_bounds[1],
                'lon_max': tile_bounds[2],
                'lat_max': tile_bounds[3],
                'nodata_frac': float(nodata_frac),
            }
            for cls_idx in range(NUM_CLASSES):
                record[f'class_{cls_idx}_frac'] = class_counts.get(cls_idx, 0) / (~nodata_mask).sum()
            
            tile_records.append(record)
            kept += 1

log.info(f"Tiling complete: {kept} kept, {discarded} discarded (>{NODATA_THRESHOLD*100:.0f}% nodata)")

In [ ]:
# Save tile metadata
tile_df = pd.DataFrame(tile_records)
tile_df.to_csv(PROCESSED_DIR / 'tile_metadata.csv', index=False)
print(f"Tile metadata saved: {len(tile_df)} tiles")
tile_df.head()

## 5. Geographic Train/Val/Test Split

In [ ]:
# Split by spatial blocks (10° × 10° lat/lon) to prevent spatial autocorrelation leakage.
# Assign each block to train/val/test, then map tiles to their block's split.

BLOCK_SIZE_DEG = 10.0

# Compute block ID for each tile based on center coordinates
tile_df['center_lon'] = (tile_df['lon_min'] + tile_df['lon_max']) / 2
tile_df['center_lat'] = (tile_df['lat_min'] + tile_df['lat_max']) / 2
tile_df['block_lon'] = (tile_df['center_lon'] // BLOCK_SIZE_DEG).astype(int)
tile_df['block_lat'] = (tile_df['center_lat'] // BLOCK_SIZE_DEG).astype(int)
tile_df['block_id'] = tile_df['block_lon'].astype(str) + '_' + tile_df['block_lat'].astype(str)

# Get unique blocks and shuffle deterministically
rng = np.random.RandomState(42)
unique_blocks = sorted(tile_df['block_id'].unique())
rng.shuffle(unique_blocks)

# Assign blocks to splits: 70/15/15
n_blocks = len(unique_blocks)
n_train = int(0.70 * n_blocks)
n_val = int(0.15 * n_blocks)

block_split = {}
for i, block in enumerate(unique_blocks):
    if i < n_train:
        block_split[block] = 'train'
    elif i < n_train + n_val:
        block_split[block] = 'val'
    else:
        block_split[block] = 'test'

tile_df['split'] = tile_df['block_id'].map(block_split)

# Log split statistics
print("Split distribution:")
for split_name in ['train', 'val', 'test']:
    subset = tile_df[tile_df['split'] == split_name]
    n = len(subset)
    pct = 100 * n / len(tile_df)
    print(f"  {split_name:5s}: {n:5d} tiles ({pct:.1f}%)")
    # Class distribution within this split
    for cls_idx in range(NUM_CLASSES):
        mean_frac = subset[f'class_{cls_idx}_frac'].mean()
        print(f"    Class {cls_idx} ({TERRAIN_CLASSES[cls_idx]:>12s}): {mean_frac:.3f}")

# Save split assignments
split_map = dict(zip(tile_df['tile_id'], tile_df['split']))
with open(SPLITS_DIR / 'split_v1.json', 'w') as f:
    json.dump(split_map, f, indent=2)

print(f"\nSplit saved to {SPLITS_DIR / 'split_v1.json'}")

In [ ]:
# Visualise the geographic split
split_colors = {'train': '#2196F3', 'val': '#FF9800', 'test': '#4CAF50'}

fig, ax = plt.subplots(figsize=(14, 6))
for split_name, color in split_colors.items():
    subset = tile_df[tile_df['split'] == split_name]
    ax.scatter(subset['center_lon'], subset['center_lat'], 
               c=color, s=2, alpha=0.5, label=f"{split_name} ({len(subset)})")

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Geographic Train/Val/Test Split')
ax.legend(markerscale=5)
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'geographic_split.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Also Tile NLDSAR Denoised Data

In [ ]:
# Look for NLDSAR rasters in data/raw/nldsar/
nldsar_dir = RAW_DIR / 'nldsar'
nldsar_files = sorted(nldsar_dir.rglob('*.tif')) if nldsar_dir.exists() else []

if nldsar_files:
    log.info(f"Found {len(nldsar_files)} NLDSAR files. Tiling to match SAR grid...")
    # TODO: reproject NLDSAR to same grid as SAR mosaic, then tile identically
    # For now, log what we have
    for f in nldsar_files:
        with rasterio.open(f) as src:
            print(f"  {f.name}: {src.width}x{src.height}, CRS={src.crs}")
else:
    log.info("No NLDSAR data found. Skipping NLDSAR tiling.")
    log.info("NLDSAR tiles can be created later when the data is available.")

## 7. Verification: Sample Tile Overlays

In [ ]:
# Randomly sample 20 tiles and visually overlay SAR + labels
sample_ids = tile_df.sample(n=min(20, len(tile_df)), random_state=42)['tile_id'].tolist()

n_samples = len(sample_ids)
n_cols_fig = 5
n_rows_fig = (n_samples + n_cols_fig - 1) // n_cols_fig

fig, axes = plt.subplots(n_rows_fig, n_cols_fig, figsize=(3 * n_cols_fig, 3 * n_rows_fig))
axes = axes.flatten()

for i, tid in enumerate(sample_ids):
    sar_tile = np.load(SAR_TILES_DIR / f"{tid}.npy")
    lbl_tile = np.load(LABEL_TILES_DIR / f"{tid}.npy")
    
    axes[i].imshow(sar_tile, cmap='gray', vmin=0, vmax=1)
    lbl_masked = np.ma.masked_where(lbl_tile == 255, lbl_tile)
    axes[i].imshow(lbl_masked, cmap=class_cmap, vmin=0, vmax=NUM_CLASSES-1, alpha=0.4)
    axes[i].set_title(tid.replace('tile_', ''), fontsize=8)
    axes[i].axis('off')

# Hide empty subplots
for i in range(n_samples, len(axes)):
    axes[i].axis('off')

plt.suptitle('Sample Tiles: SAR + Label Overlay', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'sample_tile_overlays.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nVerification complete. {len(tile_df)} tiles saved.")